### Transformacion del archivo "movie_genre.json"

In [0]:
%run "../../utilerias/funciones_comunes"

In [0]:
%run "../../utilerias/configurationes"

#### Paso 1 - Leer el archivo "movie_genre.json" usando "DataFrameReader" de spark

In [0]:
dbutils.widgets.text("param_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("param_file_date")


In [0]:
#Importamos tipos de datos
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
#Leer los datos
movie_genre_df = spark.read.table("smartdata_proyect_bronze.movies_genres").filter(f"file_date = '{v_file_date}'") 

#### Paso 2 - Renombrar las columnas y añadir nuevas columnas
 1. "movieId" renombrar a "movie_id" 
 2. "genreId" renombrar a "genre_id" 
 3. Agrega las columnas "ingestion_date" y "enviroment" 

In [0]:
#Importaciones
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
movie_genre_df_final = movie_genre_df.withColumnsRenamed({"movieId": "movie_id", "genreId": "genre_id"})

#### Paso 3 - Escribir la salida en un formato "Delta Table"[](url) con PartitionBy basado en movieId

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id  AND tgt.file_date = src.file_date'
merge_delta_lake("smartdata_proyect_silver","movies_genres",movie_genre_df_final, merge_condition, "file_date")

In [0]:
%sql
select count(1), file_date from smartdata_proyect_silver.movies_genres group by file_date;

In [0]:
dbutils.notebook.exit("success")